In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Tetapkan tahap pengoptimuman Transpiler

<details>
<summary><b>Versi pakej</b></summary>

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan penggunaan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Peranti kuantum sebenar terdedah kepada hingar dan ralat Gate, jadi mengoptimumkan Circuit untuk mengurangkan kedalaman dan bilangan Gate boleh meningkatkan keputusan yang diperoleh daripada pelaksanaan Circuit tersebut dengan ketara.
Fungsi [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) mempunyai satu argumen kedudukan yang diperlukan, `optimization_level`, yang mengawal sejauh mana usaha yang dihabiskan Transpiler untuk mengoptimumkan Circuit. Argumen ini boleh berupa integer yang mengambil salah satu nilai 0, 1, 2, atau 3.
Tahap pengoptimuman yang lebih tinggi menghasilkan Circuit yang lebih dioptimumkan dengan kos masa kompilasi yang lebih lama.
Jadual berikut menerangkan pengoptimuman yang dilakukan dengan setiap tetapan.

<table>
  <thead>
    <tr>
      <th>Tahap Pengoptimuman</th>
      <th>Penerangan</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>0</td>
      <td>
        Tiada pengoptimuman: biasanya digunakan untuk pencirian perkakasan
        - Terjemahan asas
        - Susun atur/Laluan: `TrivialLayout`, di mana ia memilih nombor Qubit fizikal yang sama seperti maya dan memasukkan SWAP untuk membuatnya berfungsi (menggunakan `SabreSwap`)
      </td>
    </tr>
    <tr>
      <td>1</td>
      <td>
        Pengoptimuman ringan:
        -   Susun atur/Laluan: Susun atur mula-mula dicuba dengan `TrivialLayout`. Jika SWAP tambahan diperlukan, susun atur dengan bilangan SWAP minimum ditemui dengan menggunakan `SabreSwap`, kemudian menggunakan `VF2LayoutPostLayout` untuk cuba memilih Qubit terbaik dalam graf.
        -   `InverseCancellation`
        -   Pengoptimuman Gate 1Q
      </td>
    </tr>
    <tr>
      <td>2</td>
      <td>
        Pengoptimuman sederhana:
          - Susun atur/Laluan: Tahap pengoptimuman 1 (tanpa trivial) + heuristik dioptimumkan dengan kedalaman carian dan percubaan fungsi pengoptimuman yang lebih besar. Memandangkan `TrivialLayout` tidak digunakan, tiada percubaan untuk menggunakan nombor Qubit fizikal dan maya yang sama.
        -   `CommutativeCancellation`
      </td>
    </tr>
    <tr>
      <td>3</td>
      <td>
        Pengoptimuman tinggi:
        - Tahap pengoptimuman 2 + heuristik dioptimumkan pada susun atur/laluan dengan lebih banyak usaha/percubaan
        - Sintesis semula blok dua Qubit menggunakan [Penguraian KAK Cartan](https://arxiv.org/abs/quant-ph/0507171).
        - Lulus yang memecahkan kesatuan:
          * `OptimizeSwapBeforeMeasure`: Menggerakkan pengukuran untuk mengelakkan SWAP
          * `RemoveDiagonalGatesBeforeMeasure`: Membuang Gate sebelum pengukuran yang tidak akan mempengaruhi pengukuran
      </td>
    </tr>
  </tbody>
</table>

## Tahap pengoptimuman dalam tindakan
Memandangkan Gate dua Qubit biasanya merupakan sumber ralat yang paling ketara, kita boleh menganggarkan "kecekapan perkakasan" transpilasi secara kasar dengan mengira bilangan Gate dua Qubit dalam Circuit yang dihasilkan.
Di sini, kita akan mencuba tahap pengoptimuman yang berbeza pada Circuit input yang terdiri daripada unitari rawak diikuti oleh Gate SWAP.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, random_unitary

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qc = QuantumCircuit(2)
qc.append(rand_U, range(2))
qc.swap(0, 1)
qc.draw("mpl", style="iqp")

<Image src="../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg)

Kita akan menggunakan Backend tiruan `FakeSherbrooke` dalam contoh kita. Pertama, mari transpil menggunakan tahap pengoptimuman 0.

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg)

Circuit yang ditranspil mempunyai enam Gate ECR dua Qubit.

Ulang untuk tahap pengoptimuman 1:

In [3]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg)

Circuit yang ditranspil masih mempunyai enam Gate ECR, tetapi bilangan Gate satu Qubit telah berkurangan.

Ulang untuk tahap pengoptimuman 2:

In [4]:
pass_manager = generate_preset_pass_manager(
    optimization_level=2, backend=backend, seed_transpiler=12345
)
qc_t2_exact = pass_manager.run(qc)
qc_t2_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg)

Ini menghasilkan keputusan yang sama seperti tahap pengoptimuman 1. Perhatikan bahawa meningkatkan tahap pengoptimuman tidak selalu membuat perbezaan.

Ulang semula, dengan tahap pengoptimuman 3:

In [5]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=12345
)
qc_t3_exact = pass_manager.run(qc)
qc_t3_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg)

Kini, hanya ada tiga Gate ECR. Kita memperoleh keputusan ini kerana pada tahap pengoptimuman 3, Qiskit cuba mensintesis semula blok Gate dua Qubit, dan mana-mana Gate dua Qubit boleh dilaksanakan menggunakan paling banyak tiga Gate ECR. Kita boleh mendapatkan lebih sedikit Gate ECR jika kita menetapkan `approximation_degree` kepada nilai kurang daripada 1, membenarkan Transpiler membuat anggaran yang mungkin memperkenalkan beberapa ralat dalam penguraian Gate (lihat [Parameter yang kerap digunakan untuk transpilasi](common-parameters#approximation-degree)):

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    approximation_degree=0.99,
    backend=backend,
    seed_transpiler=12345,
)
qc_t3_approx = pass_manager.run(qc)
qc_t3_approx.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg)

Circuit ini hanya mempunyai dua Gate ECR, tetapi ia adalah Circuit anggaran. Untuk memahami bagaimana kesannya berbeza daripada Circuit tepat, kita boleh mengira kesetiaan antara operator unitari yang dilaksanakan oleh Circuit ini, dan unitari tepat. Sebelum melakukan pengiraan, kita terlebih dahulu mengurangkan Circuit yang ditranspil, yang mengandungi 127 Qubit, kepada Circuit yang hanya mengandungi Qubit aktif, iaitu dua.

In [7]:
import numpy as np


def trace_to_fidelity_2q(trace: float) -> float:
    return (4.0 + trace * trace.conjugate()) / 20.0


# Reduce circuits down to 2 qubits so they are easy to simulate
qc_t3_exact_small = QuantumCircuit.from_instructions(qc_t3_exact)
qc_t3_approx_small = QuantumCircuit.from_instructions(qc_t3_approx)

# Compute the fidelity
exact_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_exact_small).adjoint().data, UU))
)
approx_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_approx_small).adjoint().data, UU))
)
print(
    f"Synthesis fidelity\nExact: {exact_fid:.3f}\nApproximate: {approx_fid:.3f}"
)

Synthesis fidelity
Exact: 1.000+0.000j
Approximate: 0.992+0.000j


Adjusting the optimization level can change other aspects of the circuit too, not just the number of ECR gates. For examples of how setting optimization level changes the layout, see [Representing quantum computers](./represent-quantum-computers).

## Next steps

<Admonition type="tip" title="Recommendations">
    - To learn more about the `generate_preset_passmanager` function, start with the [Transpilation default settings and configuration options](defaults-and-configuration-options) topic.
    - Continue learning about transpilation with the [Transpiler stages](transpiler-stages) topic.
    - Try the [Compare transpiler settings](/docs/guides/circuit-transpilation-settings) guide.
    - Try the [Build repetition codes](/docs/tutorials/repetition-codes) tutorial.
    - See the [Transpile API documentation.](/docs/api/qiskit/transpiler)
</Admonition>